In [ ]:
!pip install xgboost
!pip install earthengine-api
!pip install geemap
!pip install rasterio
!pip install pydrive

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
from sklearn import datasets
from sklearn.svm import SVC
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Authenticate and initialize Google Earth Engine (GEE)
import ee
ee.Authenticate()
ee.Initialize(project='ee-mohammadkhorand8989')

# Select the region of interest (ROI)
shapefile_path = 'projects/ee-mohammadkhorand8989/assets/sistan'
roi = ee.FeatureCollection(shapefile_path)

# Define the start and end time period
year = 2024
Time_start = f'{year}-08-01'  # Replace this value with your desired start date
Time_end = f'{year}-08-15'    # Replace this value with your desired end date

# Mount Google Drive to access files
from google.colab import drive
drive.mount('/content/drive')

# Define output folder and input file path
folder = f'sistan2/{year}'
shapefile_path_d = f"/content/drive/MyDrive/colab/sistan2/2015/sampled_points_with_groups-sistan.xlsx"


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
import seaborn as sns

# Read data from Excel file
df = pd.read_excel("/content/drive/MyDrive/colab/First_data/test-wind-erosion .xlsx")

# Extract relevant columns
Erosion_Rate = df.iloc[:, 5]
sand = df.iloc[:, 2]
wind = df.iloc[:, 1]
clay = df.iloc[:, 3]
silt = df.iloc[:, 4]

# Display the dataframe
print(df)

# Plot relationships between features and erosion rate
plt.figure("soil")
plt.subplot(2, 4, 1)
plt.plot(wind, Erosion_Rate, "ro")
plt.title("wind")
plt.subplot(2, 4, 2)
plt.plot(sand, Erosion_Rate, 'bo')
plt.title("sand")
plt.subplot(2, 4, 3)
plt.plot(clay, Erosion_Rate, "go")
plt.title("clay")
plt.subplot(2, 4, 4)
plt.plot(silt, Erosion_Rate, "b*")
plt.title("silt")
plt.subplot(2, 4, 5)
plt.plot(Erosion_Rate, Erosion_Rate, "r*")
plt.title("Erosion Rate")
plt.show()

# Prepare features and target variables
X = df.iloc[:, :-1].values  # All columns except the last one
y = df.iloc[:, -1].values   # Erosion Rate column (last column)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Grid search for best SVR hyperparameters
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.01, 0.1, 1, 10],
    'kernel': ['rbf']
}
grid_search = GridSearchCV(SVR(), param_grid, cv=5, scoring='r2', verbose=1)
grid_search.fit(X_train, y_train)

# Extract the best SVR model
best_model = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)

# Plot correlation matrix using seaborn
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

# Plot distributions for all features except Erosion Rate
plt.figure(figsize=(12, 6))
for i, column in enumerate(df.columns[:-1]):
    plt.subplot(2, 3, i + 1)
    sns.histplot(df[column], kde=True)
    plt.title(f'Distribution of {column}')
plt.tight_layout()
plt.show()

# Check for outliers using boxplot
sns.boxplot(data=df)
plt.title("Boxplot of All Features")
plt.xticks(rotation=45)
plt.show()

# Random Forest modeling
from sklearn.ensemble import RandomForestRegressor

# Split data again for Random Forest (in case features changed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and fit Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict and evaluate performance
y_pred_rf = rf_model.predict(X_test)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest - MSE:", mse_rf)
print("Random Forest - R2:", r2_rf)

# Plot Random Forest prediction results
plt.scatter(y_test, y_pred_rf, color='blue', label="Predicted vs Actual")
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', label="Ideal Fit")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.title("Random Forest Regression Results")
plt.show()


In [ ]:
# Extract feature importance values from the trained Random Forest model
feature_importances = rf_model.feature_importances_

# Get feature names (all columns except the target variable)
features = df.columns[:-1]

# Plot feature importance using a bar chart
plt.figure(figsize=(8, 5))
sns.barplot(x=feature_importances, y=features, palette="viridis")

# Label axes and title
plt.xlabel("Feature Importance")
plt.ylabel("Features")
plt.title("Feature Importance in Random Forest Model")

# Display the plot
plt.show()

In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

# Read data from the Excel file
df = pd.read_excel("/content/drive/MyDrive/colab/First_data/test-wind-erosion .xlsx")

# Extract features (all columns except the last) and target variable (Erosion Rate)
X = df.iloc[:, :-1].values  # All columns except the last column
y = df.iloc[:, -1].values   # Target column for Erosion Rate

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the feature data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Perform grid search to find the best parameters for the SVR model
param_grid = {
    'C': [0.1, 1, 10, 100],  # Regularization parameter
    'gamma': [0.01, 0.1, 1, 10],  # Kernel coefficient
    'kernel': ['rbf']  # Radial basis function kernel
}
grid_search = GridSearchCV(SVR(), param_grid, cv=5, scoring='r2', verbose=1)
grid_search.fit(X_train, y_train)

# Retrieve the best SVR model from grid search
best_model = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)

# Make predictions using the optimized model
y_pred_svr = best_model.predict(X_test)

# Evaluate the model performance
mse_svr = mean_squared_error(y_test, y_pred_svr)
r2_svr = r2_score(y_test, y_pred_svr)
print("SVR - MSE:", mse_svr)
print("SVR - R2:", r2_svr)

# Display a scatter plot comparing actual and predicted values
plt.scatter(y_test, y_pred_svr, color='blue', label="Predicted vs Actual")
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', label="Ideal Fit")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.title("SVR Regression Results")
plt.show()


In [ ]:
import ee
import matplotlib.pyplot as plt

# Select the region of interest (ROI)
# shapefile_path = 'projects/ee-mohammadkhorand8989/assets/fars'
roi = ee.FeatureCollection(shapefile_path)

start_year = 2015

# Define datasets
soil_moisture = ee.ImageCollection("NASA/FLDAS/NOAH01/C/GL/M/V001").select("SoilMoi00_10cm_tavg")
wind_speed = ee.ImageCollection("NASA/FLDAS/NOAH01/C/GL/M/V001").select("Wind_f_tavg")

# Divide each month into two 15‑day periods
time_ranges = []
for month in range(1, 13):
    time_ranges.append((f"{month:02d}-01", f"{month:02d}-15"))
    time_ranges.append((f"{month:02d}-16", f"{month:02d}-{30 if month in [4, 6, 9, 11] else 31 if month != 2 else 28}"))

# Function to calculate mean or maximum values for specified time intervals
def calculate_stats(collection, time_ranges, roi, start_year, stat="mean"):
    stats = []
    for start_day, end_day in time_ranges:
        start = f"{start_year}-{start_day}"
        end = f"{start_year}-{end_day}"
        date_range = ee.Filter.date(start, end)

        if stat == "mean":
            # Calculate mean value in the region
            result = (
                collection.filter(date_range)
                .mean()
                .reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=roi,
                    scale=30,
                    maxPixels=1e13
                )
                .getInfo()
            )

        elif stat == "max":
            # Calculate maximum value in the region
            result = (
                collection.filter(date_range)
                .max()
                .reduceRegion(
                    reducer=ee.Reducer.max(),
                    geometry=roi,
                    scale=30,
                    maxPixels=1e13
                )
                .getInfo()
            )

        stats.append(result)

    return stats

# Calculate mean soil moisture, mean wind speed, and maximum wind speed
soil_moisture_means = calculate_stats(soil_moisture, time_ranges, roi, start_year, stat="mean")
wind_speed_means = calculate_stats(wind_speed, time_ranges, roi, start_year, stat="mean")
wind_speed_max = calculate_stats(wind_speed, time_ranges, roi, start_year, stat="max")

# Prepare data for plotting
time_labels = [f"{month:02d}-{half}" for month in range(1, 13) for half in ["H1", "H2"]]

soil_moisture_means = [x.get("SoilMoi00_10cm_tavg", None) for x in soil_moisture_means]
wind_speed_means = [x.get("Wind_f_tavg", None) for x in wind_speed_means]
wind_speed_max = [x.get("Wind_f_tavg", None) for x in wind_speed_max]

# Calculate ratio of wind speed to soil moisture
ratio = [
    wind / soil if soil and wind else None
    for wind, soil in zip(wind_speed_means, soil_moisture_means)
]

# Calculate ratio using maximum wind speed
ratio_2 = [
    wind / soil if soil and wind else None
    for wind, soil in zip(wind_speed_max, soil_moisture_means)
]

# Plot results using multiple subplots
plt.figure(figsize=(16, 16))

# Plot 1: Soil moisture
plt.subplot(5, 1, 1)
plt.plot(time_labels, soil_moisture_means, label="Soil Moisture (0–10 cm)", marker="o", color="blue")
plt.xlabel("Time Period (15‑day intervals)")
plt.ylabel("Soil Moisture")
plt.title(f"15‑Day Mean Soil Moisture ({start_year})")
plt.xticks(rotation=45)
plt.legend()
plt.grid()

# Plot 2: Mean wind speed
plt.subplot(5, 1, 2)
plt.plot(time_labels, wind_speed_means, label="Mean Wind Speed", marker="s", color="green")
plt.xlabel("Time Period (15‑day intervals)")
plt.ylabel("Wind Speed")
plt.title("15‑Day Mean Wind Speed")
plt.xticks(rotation=45)
plt.legend()
plt.grid()

# Plot 3: Ratio of wind speed to soil moisture
plt.subplot(5, 1, 3)
plt.plot(time_labels, ratio, label="Wind Speed / Soil Moisture", marker="^", color="red")
plt.xlabel("Time Period (15‑day intervals)")
plt.ylabel("Ratio (Wind / Soil Moisture)")
plt.title("Ratio of Wind Speed to Soil Moisture")
plt.xticks(rotation=45)
plt.legend()
plt.grid()

# Plot 4: Maximum wind speed
plt.subplot(5, 1, 4)
plt.plot(time_labels, wind_speed_max, label="Maximum Wind Speed", marker="s", color="green")
plt.xlabel("Time Period (15‑day intervals)")
plt.ylabel("Wind Speed")
plt.title("15‑Day Maximum Wind Speed")
plt.xticks(rotation=45)
plt.legend()
plt.grid()

# Plot 5: Ratio of maximum wind speed to soil moisture
plt.subplot(5, 1, 5)
plt.plot(time_labels, ratio_2, label="Max Wind Speed / Soil Moisture", marker="*", color="red")
plt.xlabel("Time Period (15‑day intervals)")
plt.ylabel("Ratio (Max Wind / Soil Moisture)")
plt.title("Ratio of Maximum Wind Speed to Soil Moisture")
plt.xticks(rotation=45)
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()


In [ ]:
# Create a DataFrame to store all calculated variables
data = {
    "Time Period": time_labels,
    "Soil Moisture": soil_moisture_means,
    "Mean Wind Speed": wind_speed_means,
    "Max Wind Speed": wind_speed_max,
    "Wind Speed / Soil Moisture": ratio,
    "Max Wind Speed / Soil Moisture": ratio_2,
}

df = pd.DataFrame(data)

# Save the DataFrame to an Excel file
excel_filename = f"/content/drive/MyDrive/colab/{folder}/wind_soil_data_{start_year}.xlsx"
df.to_excel(excel_filename, index=False, sheet_name="Data")

# Confirm file saving
print(f"Data saved to {excel_filename}")


In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# Load the shapefile
# shapefile_path_d = '/content/drive/MyDrive/colab/UrmiaBasin.shp'
# shapefile_path_d = '/content/drive/MyDrive/colab/shape file/Isfahan/Isfahan.shx'
gdf = gpd.read_file(shapefile_path_d)

# Assuming the geometry contains a Polygon
polygon = gdf.geometry.iloc[0]  # Use the first geometry

# Get the bounding box of the polygon
minx, miny, maxx, maxy = polygon.bounds

# Define pixel size (in meters or degrees)
pixel_size = 0.0025  # In degrees (0.01 degrees is approximately 1.1 km)

# Generate pixel center coordinates within the bounding box
# Start from the center of the first pixel
x_coords = np.arange(minx + pixel_size / 2, maxx, pixel_size)
y_coords = np.arange(miny + pixel_size / 2, maxy, pixel_size)
grid_points = [Point(x, y) for x in x_coords for y in y_coords]

# Filter points that are inside the polygon
points_in_polygon = [point for point in grid_points if polygon.contains(point)]

# Convert the filtered points to a GeoDataFrame
points_gdf = gpd.GeoDataFrame(geometry=points_in_polygon)

# Extract coordinates of the pixel centers
pixels_df = pd.DataFrame({
    'Longitude': [point.x for point in points_gdf.geometry],
    'Latitude': [point.y for point in points_gdf.geometry]
})

# Display the first few pixels
print(pixels_df.head())

# Save the pixel coordinates to a CSV file
pixels_df.to_csv("/content/drive/MyDrive/colab/Urumiaya_basin/pixels_in_polygon.csv", index=False)
print("Pixels saved to pixels_in_polygon.csv") # Corrected print statement

# Display a map of the pixel centers within the polygon
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 10))
plt.scatter(pixels_df['Longitude'], pixels_df['Latitude'], s=1, color='blue', alpha=0.5)
plt.title("Pixel Centers in Polygon")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()


In [ ]:
# Data Grouping

# Define the path for the input CSV file
input_csv = f"/content/drive/MyDrive/colab/Hamoon/PixelCoordinates-{folder}.csv"

# Read the CSV file
pixels_df = pd.read_csv(input_csv)

# Add an 'ID' column starting from 1
pixels_df['ID'] = pixels_df.index + 1

# Split data into groups of 4000 points each
pixels_df['Group'] = (pixels_df.index // 4000) + 1

# Save the output to an Excel file
output_file_Group = "/content/drive/MyDrive/colab/Hamoon/sampled_points_with_groups-HAMOON.xlsx"
pixels_df.to_excel(output_file_Group, index=False)

# Print a confirmation message for file saving
print(f"Data has been saved to {output_file_Group}")

# Calculate the total number of groups
number_of_groups = pixels_df['Group'].nunique()
print(f"Number of Groups: {number_of_groups}")

# Generate distinct colors for each group using a colormap
colors = plt.cm.get_cmap('tab20', len(pixels_df['Group'].unique()))

# Plot the points, coloring them based on their group
plt.figure(figsize=(10, 10))
for group, group_data in pixels_df.groupby('Group'):
    plt.scatter(
        group_data['Longitude'],
        group_data['Latitude'],
        s=1,
        color=colors(group - 1),
        label=f'Group {group}',
        alpha=0.7
    )

# Add chart details (title, labels, and legend)
plt.title("Pixel Centers in CSV by Group")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend(markerscale=10, loc='upper right', fontsize='small')
plt.show()


In [ ]:
import ee
import geemap
import pandas as pd
import math
from google.colab import drive
import folium

# Authenticate and initialize Google Earth Engine (GEE)
ee.Authenticate()
ee.Initialize(project='ee-mohammadkhorand8989')

# Mount Google Drive in Colab
drive.mount('/content/drive')

# Path to the Excel file and time range
# shapefile_path_d = "/content/drive/MyDrive/colab/fars/sampled_points_with_groups-fars.xlsx"
# Time_start = '2023-08-01'
# Time_end = '2023-08-15'

# Load Excel data containing coordinates and group IDs
data = pd.read_excel(shapefile_path_d)

# Define wind speed dataset from GEE (GLDAS)
wind_speed = (ee.ImageCollection('NASA/GLDAS/V021/NOAH/G025/T3H')
              .filterDate(Time_start, Time_end)
              .select('Wind_f_inst')
              .max())

# Surface roughness parameters (based on terrain type)
z0 = 0.03  # roughness length for smooth terrain
z_target = 2  # target height (2 m)
z_initial = 10  # initial height (10 m, satellite data)

# Create a list to store wind data for all groups
all_groups_data_wind = []

# Create a Folium map
map_center = [data["Latitude"].mean(), data["Longitude"].mean()]
m = folium.Map(location=map_center, zoom_start=6)

# Create a layer to store wind points
wind_layer = folium.FeatureGroup(name="Wind Speed Data")

# Total number of groups
number_of_groups = data['Group'].max()

# Loop through all groups
for group_number in range(1, number_of_groups + 1):
    print(f"Processing Group {group_number}...")

    # Select points belonging to the current group
    group_data = data[data['Group'] == group_number]

    # Create GEE points from the coordinates
    points = [ee.Geometry.Point([lon, lat]) for lon, lat in zip(group_data['Longitude'], group_data['Latitude'])]

    # Create a FeatureCollection from the points
    sample_points = ee.FeatureCollection([ee.Feature(point) for point in points])

    # Extract wind speed values at the specified points
    extracted_values = wind_speed.sampleRegions(
        collection=sample_points,
        scale=250,
        geometries=True
    ).getInfo()

    # Process the extracted data
    latitudes = []
    longitudes = []
    wind_speeds_10m = []

    for feature in extracted_values['features']:
        coordinates = feature['geometry']['coordinates']
        value = feature['properties']['Wind_f_inst']
        latitudes.append(coordinates[1])
        longitudes.append(coordinates[0])
        wind_speeds_10m.append(value)

    # Calculate wind speed at 2 m height using the logarithmic wind profile
    wind_speeds_2m = [
        ws * (math.log(z_target / z0) / math.log(z_initial / z0)) if ws else None
        for ws in wind_speeds_10m
    ]

    # Create a DataFrame for the current group
    group_df = pd.DataFrame({
        'Group': group_number,
        'Latitude': latitudes,
        'Longitude': longitudes,
        'WindSpeed': wind_speeds_2m
    })

    # Add points to the map layer
    for lat, lon, ws in zip(latitudes, longitudes, wind_speeds_2m):
        folium.CircleMarker(
            location=[lat, lon],
            radius=5 + (ws * 0.5),  # circle size based on wind speed
            color="blue",
            fill=True,
            fill_color="blue",
            fill_opacity=0.7,
            popup=f"Wind Speed: {

In [ ]:
from google.colab import drive
import requests
from PIL import Image
from io import BytesIO


# Authenticate and initialize Google Earth Engine (GEE)
ee.Authenticate()
ee.Initialize(project='ee-mohammadkhorand8989')
all_groups_data_sand = []

# Select the region of interest (ROI) as a GEE FeatureCollection
roi = ee.FeatureCollection(shapefile_path)

# Load Excel data containing coordinates and group IDs
data = pd.read_excel(shapefile_path_d)

# Sand content band (b0) from OpenLandMap
sand = (ee.Image("OpenLandMap/SOL/SOL_SAND-WFRACTION_USDA-3A1A1A_M/v02")
        .select('b0')
        .clip(roi))

# Loop through all groups (1 to number_of_groups)
for group_number in range(1, number_of_groups + 1):
    print(f"Processing Group {group_number}...")

    # Select points belonging to the current group
    group_data = data[data['Group'] == group_number]

    # Create GEE points from the group's coordinates
    points = [ee.Geometry.Point([lon, lat]) for lon, lat in zip(group_data['Longitude'], group_data['Latitude'])]

    # Create a FeatureCollection from the points
    sample_points = ee.FeatureCollection([ee.Feature(point) for point in points])

    # Extract sand fraction for the specified points
    extracted_values = sand.sampleRegions(
        collection=sample_points,
        scale=1000,
        geometries=True
    ).getInfo()

    # Process extracted data
    ids = []
    latitudes = []
    longitudes = []
    sand_values = []

    for feature in extracted_values['features']:
        coordinates = feature['geometry']['coordinates']
        value = feature['properties']['b0']  # Make sure the band name is correct
        ids.append(feature['id'])
        latitudes.append(coordinates[1])
        longitudes.append(coordinates[0])
        sand_values.append(value)

    # Create a DataFrame for the current group
    group_df = pd.DataFrame({
        'Group': group_number,
        'ID': ids,
        'Latitude': latitudes,
        'Longitude': longitudes,
        'Sand': sand_values
    })

    # Add the DataFrame to the list for all groups
    all_groups_data_sand.append(group_df)

# Merge all group DataFrames into one
final_df = pd.concat(all_groups_data_sand, ignore_index=True)

# Save the data to an Excel file
# folder = 'Hamoon'
output_path_sand = f'/content/drive/MyDrive/colab/{folder}/all_groups_sand_fraction.xlsx'
final_df.to_excel(output_path_sand, index=False)
print(f"Final Excel file saved: {output_path_sand}")

# Visualize sand map (optional: you can select a specific group for display if needed)
sand_visualization = sand.visualize(min=0, max=100, palette=[
    '#ffff00', '#f8f806', '#f1f10c', '#ebeb13', '#e4e419', '#dddd20',
    '#d7d726', '#d0d02d', '#caca33', '#bcbc41', '#b6b647', '#b0b04e',
    '#a9a954', '#a3a35a', '#9c9c61', '#959568', '#8f8f6e', '#898975',
    '#82827b', '#7b7b82', '#757589', '#6e6e8f', '#686895', '#61619c',
    '#5a5aa3', '#5454a9', '#4d4db0', '#4747b6', '#4141bc', '#3a3ac3',
    '#3333ca', '#2d2dd0', '#2626d7', '#2020dd', '#1919e4', '#1212eb',
    '#0c0cf1', '#0606f8', '#0000ff'
])
url = sand_visualization.getThumbURL({'region': roi.geometry(), 'dimensions': 512, 'format': 'png'})

# Download and display the image
response = requests.get(url)
img = Image.open(BytesIO(response.content))

# Create matplotlib figure and legend
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(img)
ax.axis('off')
ax.set_title("Sand Fraction Visualization")

# Setup colormap and legend for sand fraction
cmap = mpl.colors.ListedColormap([
    '#ffff00', '#f8f806', '#f1f10c', '#ebeb13', '#e4e419', '#dddd20',
    '#d7d726', '#d0d02d', '#caca33', '#bcbc41', '#b6b647', '#b0b04e',
    '#a9a954', '#a3a35a', '#9c9c61', '#959568', '#8f8f6e', '#898975',
    '#82827b', '#7b7b82', '#757589', '#6e6e8f', '#686895', '#61619c',
    '#5a5aa3', '#5454a9', '#4d4db0', '#4747b6', '#4141bc', '#3a3ac3',
    '#3333ca', '#2d2dd0', '#2626d7', '#2020dd', '#1919e4', '#1212eb',
    '#0c0cf1', '#0606f8', '#0000ff'
])
norm = mpl.colors.Normalize(vmin=0, vmax=100)

# Add colorbar
cbar = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
    ax=ax, orientation='vertical', fraction=0.046, pad=0.04
)
cbar.set_label('Sand Fraction (%)')

plt.show()


In [ ]:
import ee
import pandas as pd
from google.colab import drive
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
import matplotlib as mpl

# Authenticate and initialize Google Earth Engine (GEE)
ee.Authenticate()
ee.Initialize(project='ee-mohammadkhorand8989')

# Select the region of interest (ROI)
roi = ee.FeatureCollection(shapefile_path)

# Load Excel data containing latitude, longitude, and group numbers
data = pd.read_excel(shapefile_path_d)

# Initialize a list to store clay data for all groups
all_groups_data_clay = []

# Load the clay fraction band from OpenLandMap
clay = (ee.Image("OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02")
        .select('b0')
        .clip(roi))

# Loop through all groups (1 to total number_of_groups)
for group_number in range(1, number_of_groups + 1):
    print(f"Processing Group {group_number}...")

    # Select points belonging to the current group
    group_data = data[data['Group'] == group_number]

    # Create GEE points from the group's coordinates
    points = [ee.Geometry.Point([lon, lat]) for lon, lat in zip(group_data['Longitude'], group_data['Latitude'])]

    # Create a FeatureCollection from the points
    sample_points = ee.FeatureCollection([ee.Feature(point) for point in points])

    # Extract clay fraction values for the specified points
    extracted_values = clay.sampleRegions(
        collection=sample_points,
        scale=1000,
        geometries=True
    ).getInfo()

    # Process the extracted features
    ids = []
    latitudes = []
    longitudes = []
    clay_values = []

    for feature in extracted_values['features']:
        coordinates = feature['geometry']['coordinates']
        value = feature['properties']['b0']  # Verify that the band name is correct
        ids.append(feature['id'])
        latitudes.append(coordinates[1])
        longitudes.append(coordinates[0])
        clay_values.append(value)

    # Create a DataFrame for the current group
    group_df = pd.DataFrame({
        'Group': group_number,
        'ID': ids,
        'Latitude': latitudes,
        'Longitude': longitudes,
        'Clay': clay_values
    })

    # Append the current group data to the main list
    all_groups_data_clay.append(group_df)

# Combine all group DataFrames into one
final_df = pd.concat(all_groups_data_clay, ignore_index=True)

# Save combined clay data to an Excel file
output_path_clay = f'/content/drive/MyDrive/colab/{folder}/all_groups_clay_fraction.xlsx'
final_df.to_excel(output_path_clay, index=False)
print(f"Final Excel file saved: {output_path_clay}")

# Visualize the clay fraction map (optional)
clay_visualization = clay.visualize(min=0, max=100, palette=[
    '#ffff00', '#f8f806', '#f1f10c', '#ebeb13', '#e4e419', '#dddd20',
    '#d7d726', '#d0d02d', '#caca33', '#bcbc41', '#b6b647', '#b0b04e',
    '#a9a954', '#a3a35a', '#9c9c61', '#959568', '#8f8f6e', '#898975',
    '#82827b', '#7b7b82', '#757589', '#6e6e8f', '#686895', '#61619c',
    '#5a5aa3', '#5454a9', '#4d4db0', '#4747b6', '#4141bc', '#3a3ac3',
    '#3333ca', '#2d2dd0', '#2626d7', '#2020dd', '#1919e4', '#1212eb',
    '#0c0cf1', '#0606f8', '#0000ff'
])
url = clay_visualization.getThumbURL({'region': roi.geometry(), 'dimensions': 512, 'format': 'png'})

# Download and display the visualization image
response = requests.get(url)
img = Image.open(BytesIO(response.content))

# Create a figure and legend using Matplotlib
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(img)
ax.axis('off')
ax.set_title("Clay Fraction Visualization")

# Define color palette and normalization for the legend
cmap = mpl.colors.ListedColormap([
    '#ffff00', '#f8f806', '#f1f10c', '#ebeb13', '#e4e419', '#dddd20',
    '#d7d726', '#d0d02d', '#caca33', '#bcbc41', '#b6b647', '#b0b04e',
    '#a9a954', '#a3a35a', '#9c9c61', '#959568', '#8f8f6e', '#898975',
    '#82827b', '#7b7b82', '#757589', '#6e6e8f', '#686895', '#61619c',
    '#5a5aa3', '#5454a9', '#4d4db0', '#4747b6', '#4141bc', '#3a3ac3',
    '#3333ca', '#2d2dd0', '#2626d7', '#2020dd', '#1919e4', '#1212eb',
    '#0c0cf1', '#0606f8', '#0000ff'
])
norm = mpl.colors.Normalize(vmin=0, vmax=100)

# Create colorbar
cbar = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
    ax=ax, orientation='vertical', fraction=0.046, pad=0.04
)
cbar.set_label('Clay Fraction (%)')

plt.show()


In [ ]:
import ee
import pandas as pd
from google.colab import drive
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
import matplotlib as mpl

# Authenticate and initialize Google Earth Engine (GEE)
ee.Authenticate()
ee.Initialize(project='')

# Initialize a list to store silt data for all groups
all_groups_data_silt = []

# Select the region of interest (ROI)
roi = ee.FeatureCollection(shapefile_path)

# Load Excel data containing coordinates and group numbers
data = pd.read_excel(shapefile_path_d)

# Load the "Clay" and "Sand" fraction bands
clay = (ee.Image("OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02")
        .select('b0')
        .clip(roi))

sand = (ee.Image("OpenLandMap/SOL/SOL_SAND-WFRACTION_USDA-3A1A1A_M/v02")
        .select('b0')
        .clip(roi))

# Compute the "Silt" fraction band: Silt = 100 − (Clay + Sand)
silt = clay.add(sand).multiply(-1).add(100).rename('Silt')

# Loop over all groups
for group_number in range(1, number_of_groups + 1):
    print(f"Processing Group {group_number}...")

    # Select points belonging to the current group
    group_data = data[data['Group'] == group_number]

    # Create GEE points from the group's coordinates
    points = [
        ee.Geometry.Point([lon, lat])
        for lon, lat in zip(group_data['Longitude'], group_data['Latitude'])
    ]

    # Create a FeatureCollection from the points
    sample_points = ee.FeatureCollection([ee.Feature(point) for point in points])

    # Extract "Silt" values for the specified points
    extracted_values = silt.sampleRegions(
        collection=sample_points,
        scale=1000,
        geometries=True
    ).getInfo()

    # Process the extracted features
    ids = []
    latitudes = []
    longitudes = []
    silt_values = []

    for feature in extracted_values['features']:
        coordinates = feature['geometry']['coordinates']
        value = feature['properties']['Silt']
        ids.append(feature['id'])
        latitudes.append(coordinates[1])
        longitudes.append(coordinates[0])
        silt_values.append(value)

    # Create a DataFrame for the current group
    group_df = pd.DataFrame({
        'Group': group_number,
        'ID': ids,
        'Latitude': latitudes,
        'Longitude': longitudes,
        'Silt': silt_values
    })

    # Append the current group data to the main list
    all_groups_data_silt.append(group_df)

# Merge all group DataFrames into a single DataFrame
final_df = pd.concat(all_groups_data_silt, ignore_index=True)

# Save the results to an Excel file
output_path_silt = f'/content/drive/MyDrive/colab/{folder}/all_groups_silt_fraction.xlsx'
final_df.to_excel(output_path_silt, index=False)
print(f"Final Excel file saved: {output_path_silt}")

# Visualize the silt fraction map (optional)
silt_visualization = silt.visualize(min=0, max=100, palette=[
    '#ffff00', '#f8f806', '#f1f10c', '#ebeb13', '#e4e419', '#dddd20',
    '#d7d726', '#d0d02d', '#caca33', '#bcbc41', '#b6b647', '#b0b04e',
    '#a9a954', '#a3a35a', '#9c9c61', '#959568', '#8f8f6e', '#898975',
    '#82827b', '#7b7b82', '#757589', '#6e6e8f', '#686895', '#61619c',
    '#5a5aa3', '#5454a9', '#4d4db0', '#4747b6', '#4141bc', '#3a3ac3',
    '#3333ca', '#2d2dd0', '#2626d7', '#2020dd', '#1919e4', '#1212eb',
    '#0c0cf1', '#0606f8', '#0000ff'
])

url = silt_visualization.getThumbURL({
    'region': roi.geometry(),
    'dimensions': 512,
    'format': 'png'
})

# Download and display the image
response = requests.get(url)
img = Image.open(BytesIO(response.content))

# Create the figure and legend
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(img)
ax.axis('off')
ax.set_title("Silt Fraction Visualization")

# Define the color palette and legend
cmap = mpl.colors.ListedColormap([
    '#ffff00', '#f8f806', '#f1f10c', '#ebeb13', '#e4e419', '#dddd20',
    '#d7d726', '#d0d02d', '#caca33', '#bcbc41', '#b6b647', '#b0b04e',
    '#a9a954', '#a3a35a', '#9c9c61', '#959568', '#8f8f6e', '#898975',
    '#82827b', '#7b7b82', '#757589', '#6e6e8f', '#686895', '#61619c',
    '#5a5aa3', '#5454a9', '#4d4db0', '#4747b6', '#4141bc', '#3a3ac3',
    '#3333ca', '#2d2dd0', '#2626d7', '#2020dd', '#1919e4', '#1212eb',
    '#0c0cf1', '#0606f8', '#0000ff'
])
norm = mpl.colors.Normalize(vmin=0, vmax=100)

# Create the colorbar
cbar = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
    ax=ax,
    orientation='vertical',
    fraction=0.046,
    pad=0.04
)
cbar.set_label('Silt Fraction (%)')

plt.show()


In [ ]:
import ee
import pandas as pd
from google.colab import drive
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
import matplotlib as mpl

# Authenticate and initialize Google Earth Engine (GEE)
ee.Authenticate()
ee.Initialize(project='')

# Select the region of interest (ROI)
roi = ee.FeatureCollection(shapefile_path)

# Initialize a list to store soil moisture data for all groups
all_groups_data_moisture = []

# Load Excel data containing coordinates and group numbers
data = pd.read_excel(shapefile_path_d)

# Define start time and end time for the analysis period

# Soil moisture band (0–10 cm depth)
soil_moisture = (
    ee.ImageCollection('NASA/GLDAS/V021/NOAH/G025/T3H')
    .filterDate(Time_start, Time_end)
    .select('SoilMoi0_10cm_inst')
    .mean()
    .clip(roi)
)

# Total number of groups
number_of_groups = 51

# Loop over all groups
for group_number in range(1, number_of_groups + 1):
    print(f"Processing Group {group_number}...")

    # Select points belonging to the current group
    group_data = data[data['Group'] == group_number]

    # Create GEE points from the group's coordinates
    points = [
        ee.Geometry.Point([lon, lat])
        for lon, lat in zip(group_data['Longitude'], group_data['Latitude'])
    ]

    # Create a FeatureCollection from the points
    sample_points = ee.FeatureCollection([ee.Feature(point) for point in points])

    # Extract soil moisture values for the specified points
    extracted_values = soil_moisture.sampleRegions(
        collection=sample_points,
        scale=1000,
        geometries=True
    ).getInfo()

    # Process the extracted features
    ids = []
    latitudes = []
    longitudes = []
    soil_moisture_values = []

    for feature in extracted_values['features']:
        coordinates = feature['geometry']['coordinates']
        value = feature['properties']['SoilMoi0_10cm_inst']
        ids.append(feature['id'])
        latitudes.append(coordinates[1])
        longitudes.append(coordinates[0])
        soil_moisture_values.append(value)

    # Create a DataFrame for the current group
    group_df = pd.DataFrame({
        'Group': group_number,
        'ID': ids,
        'Latitude': latitudes,
        'Longitude': longitudes,
        'Soil_Moisture': soil_moisture_values
    })

    # Append the current group data to the main list
    all_groups_data_moisture.append(group_df)

# Merge all group DataFrames into a single DataFrame
final_df = pd.concat(all_groups_data_moisture, ignore_index=True)

# Save the results to an Excel file
output_path = f'/content/drive/MyDrive/colab/{folder}/all_groups_soil_moisture-1.xlsx'
final_df.to_excel(output_path, index=False)
print(f"Final Excel file saved: {output_path}")

# Visualize the soil moisture map (optional)
soil_moisture_visualization = soil_moisture.visualize(
    min=0,
    max=100,
    palette=['blue', 'cyan', 'green', 'yellow', 'orange', 'red']
)

url = soil_moisture_visualization.getThumbURL({
    'region': roi.geometry(),
    'dimensions': 512,
    'format': 'png'
})

# Download and display the image
response = requests.get(url)
img = Image.open(BytesIO(response.content))

# Create the figure and legend
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(img)
ax.axis('off')
ax.set_title("Soil Moisture Visualization")

# Define the color palette and legend
cmap = mpl.colors.ListedColormap(['blue', 'cyan', 'green', 'yellow', 'orange', 'red'])
norm = mpl.colors.Normalize(vmin=0, vmax=100)

# Create the colorbar
cbar = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
    ax=ax,
    orientation='vertical',
    fraction=0.046,
    pad=0.04
)
cbar.set_label('Soil Moisture (%)')

plt.show()


In [ ]:
# Nearest-distance matching with a threshold of 0.0025 degrees

# Load input files
band_1_df = pd.read_excel(f'/content/drive/MyDrive/colab/{folder}/all_groups_wind_speed.xlsx')
band_2_df = pd.read_excel(f'/content/drive/MyDrive/colab/sistan2/2015/all_groups_sand_fraction.xlsx')
band_3_df = pd.read_excel(f'/content/drive/MyDrive/colab/sistan2/2015/all_groups_clay_fraction.xlsx')
band_4_df = pd.read_excel(f'/content/drive/MyDrive/colab/sistan2/2015/all_groups_silt_fraction.xlsx')
soil_moisture_df = pd.read_excel(f'/content/drive/MyDrive/colab/{folder}/all_groups_soil_moisture-1.xlsx')

# Define the main columns
final_df = pd.DataFrame({
    'Latitude': band_1_df['Latitude'],   # Latitude column from band_1 file
    'Longitude': band_1_df['Longitude'], # Longitude column from band_1 file
    'test_duration': [10] * len(band_1_df)  # Constant value for all rows
})

# Function to find the nearest value within a distance threshold
def get_nearest_value_with_threshold(lat, lon, source_df, lat_col='Latitude', lon_col='Longitude', value_col='Value', threshold=0.0025):
    distances = np.sqrt((source_df[lat_col] - lat) ** 2 + (source_df[lon_col] - lon) ** 2)
    min_distance = distances.min()
    if min_distance <= threshold:
        nearest_row = source_df.iloc[distances.idxmin()]
        return nearest_row[value_col]
    return 0  # If the minimum distance exceeds the threshold, return zero

# Function to retrieve values based on exact or nearest coordinate match
def match_value(lat, lon, source_df, lat_col='Latitude', lon_col='Longitude', value_col='Value', allow_nearest=False, threshold=0.0025):
    row = source_df[(source_df[lat_col] == lat) & (source_df[lon_col] == lon)]
    if not row.empty:
        return row.iloc[0][value_col]
    elif allow_nearest:
        return get_nearest_value_with_threshold(lat, lon, source_df, lat_col, lon_col, value_col, threshold)
    return None

# Add additional columns using the matching functions
final_df['wind_speed'] = final_df.apply(lambda row: match_value(row['Latitude'], row['Longitude'], band_1_df, value_col='WindSpeed'), axis=1)
final_df['sand'] = final_df.apply(lambda row: match_value(row['Latitude'], row['Longitude'], band_2_df, value_col='Sand', allow_nearest=True), axis=1)
final_df['clay'] = final_df.apply(lambda row: match_value(row['Latitude'], row['Longitude'], band_3_df, value_col='Clay', allow_nearest=True), axis=1)
final_df['silt'] = final_df.apply(lambda row: match_value(row['Latitude'], row['Longitude'], band_4_df, value_col='Silt', allow_nearest=True), axis=1)
final_df['moisture'] = final_df.apply(lambda row: match_value(row['Latitude'], row['Longitude'], soil_moisture_df, value_col='Soil_Moisture'), axis=1)

# Display the DataFrame
print(final_df)

# Save output to CSV
final_df.to_csv(f'/content/drive/MyDrive/colab/{folder}/final_soil_data.csv', index=False)
print(f"Final DataFrame saved as 'final_soil_data {year}.csv'")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
import seaborn as sns
import ee
import requests
from PIL import Image
from io import BytesIO

# Load the new dataset
final_soil_data = pd.read_csv('')

# Separate features and target variables
X_new = final_soil_data[['Latitude', 'Longitude', 'sand', 'clay', 'silt', 'moisture']].values

# Standardize the features
scaler = StandardScaler()
X_new_scaled = scaler.fit_transform(X_new)

# Load/Initialize the best models (from the previous stage)
best_rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
# Train the model using the previously defined training data
best_rf_model.fit(X_train, y_train)  

# Predict the erosion rate using the trained model
predictions_rf = best_rf_model.predict(X_new_scaled)

# Add the 'Erosion_Rate' predictions as a new column to the DataFrame
final_soil_data['Erosion_Rate'] = predictions_rf

# Save the updated dataset including the calculated Erosion_Rate
final_soil_data.to_csv('/content/drive/MyDrive/colab/Urumiaya_basin/final_soil_data_with_erosion-test-3.csv', index=False)
print("Erosion Rate calculated and saved successfully.")

# Visualize the results
plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=final_soil_data, 
    x='Longitude', 
    y='Latitude', 
    hue='Erosion_Rate', 
    palette='viridis', 
    size='Erosion_Rate', 
    sizes=(20, 200)
)
plt.title('Predicted Wind Erosion Rate Map')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Note: plt.colorbar requires a scalar mappable. 
# In Seaborn scatterplots, the 'hue' usually creates a legend instead.
plt.show()


In [ ]:
import pandas as pd

# Load the CSV file
file_path = f'/content/drive/MyDrive/colab/{folder}/final_soil_data_with_erosion.csv'
df = pd.read_csv(file_path)

# Display the first few rows to understand the structure
df.head()

# Check whether 'Erosion_Rate' column exists
if 'Erosion_Rate' not in df.columns:
    print("Error: 'Erosion_Rate' column not found in the DataFrame.")
else:
    # Define quantile‑based bins for erosion classification
    df['Erosion Class'] = pd.qcut(
        df['Erosion_Rate'],
        q=5,
        labels=["Very Low", "Low", "Moderate", "High", "Very High"]
    )

    # Save the categorized data to a new Excel file
    output_file_path = f"/content/drive/MyDrive/colab/{folder}/categorized_wind_erosion.xlsx"
    df.to_excel(output_file_path, index=False)

    # Return the output path
    output_file_path


import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load categorized erosion data
data_path = f"/content/drive/MyDrive/colab/{folder}/categorized_wind_erosion.xlsx"
df = pd.read_excel(data_path)

# Check necessary columns
if 'Erosion_Rate' not in df.columns:
    print("Error: 'Erosion_Rate' column not found in the DataFrame.")
if 'Erosion Class' not in df.columns:
    print("Error: 'Erosion Class' column not found in the DataFrame.")
else:
    # Define colors for erosion categories
    color_map = {
        'Very Low': 'white',
        'Low': 'green',
        'Moderate': 'blue',
        'High': 'yellow',
        'Very High': 'red'
    }

    # Plot the erosion‑risk scatter map
    plt.figure(figsize=(10, 8))
    scatter = sns.scatterplot(
        data=df,
        x='Longitude',
        y='Latitude',
        hue='Erosion Class',
        palette=color_map,
        size='Erosion Class',
        sizes=(10, 10)
    )

    plt.title(f'Wind Erosion Risk Map {year}')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.legend(title='Erosion Risk Category')

    # Save before showing (safer)
    plt.savefig(
        f'/content/drive/MyDrive/colab/{folder}/Wind Erosion Risk Map {year}.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.show()
